03.4 Automated ETL Pipeline: Build labeled_step_test

In [0]:
### Load raw tables, normalize columns, label step events, and build final_df ###

from pyspark.sql.functions import col, lit, when, regexp_extract, length, expr

df_device = spark.table("workspace.bronze.device_messages_raw")
df_steps  = spark.table("workspace.bronze.rapid_step_tests_raw")

def rename_if_exists(df, old, new):
    return df.withColumnRenamed(old, new) if old in df.columns else df

d = df_device
d = rename_if_exists(d, "deviceId", "device_id")
d = rename_if_exists(d, "sensorType", "sensor_type")
d = d.withColumn("timestamp", expr("timestamp_millis(timestamp)"))

dist_digits = regexp_extract(col("distance"), r"(\d+)", 1)
d = d.withColumn(
    "distance_cm",
    when(length(dist_digits) > 0, dist_digits.cast("int")).otherwise(lit(None).cast("int"))
)

s = df_steps
s = rename_if_exists(s, "deviceId", "device_id")
s = rename_if_exists(s, "startTime", "start_time")
s = rename_if_exists(s, "stopTime", "stop_time")
s = s.withColumn("start_time", expr("timestamp_millis(start_time)"))
s = s.withColumn("stop_time",  expr("timestamp_millis(stop_time)"))


# Build step windows with non-colliding key name
s_win = (
    s.select("device_id", "start_time", "stop_time")
     .dropna(subset=["device_id", "start_time", "stop_time"])
     .withColumnRenamed("device_id", "step_device_id")
)

# Join + label
labeled = (
    d.alias("d")
     .join(
         s_win.alias("s"),
         (col("d.device_id") == col("s.step_device_id")) &
         (col("d.timestamp").between(col("s.start_time"), col("s.stop_time"))),
         "left"
     )
     .withColumn(
         "step_label",
         when(col("s.start_time").isNotNull(), lit("step")).otherwise(lit("no_step"))
     )
     .withColumn("source_label", lit("device"))
)

# Final curated dataframe expected by the assignment queries
final_df = labeled.select(
    "timestamp",
    "sensor_type",
    "distance_cm",
    "device_id",
    "step_label",
    "source_label"
)

# Make it available to SQL as final_df (their instructions require this pattern)
final_df.createOrReplaceTempView("final_df")


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.labeled_step_test AS
SELECT * FROM final_df;


# Verification

In [0]:
%sql
SELECT step_label, COUNT(*)
FROM workspace.silver.labeled_step_test
GROUP BY step_label;


In [0]:
%sql
SELECT *
FROM workspace.silver.labeled_step_test
WHERE step_label NOT IN ('step','no_step')
   OR step_label IS NULL
LIMIT 50;


In [0]:
%sql
SELECT source_label, COUNT(*)
FROM workspace.silver.labeled_step_test
GROUP BY source_label;


In [0]:
%sql
SELECT *
FROM workspace.silver.labeled_step_test
WHERE source_label NOT IN ('device','step')
   OR source_label IS NULL
LIMIT 50;


# Ethics
When automating health-related data pipelines, engineers must consider protecting the people behind the data, not solely the data. There are organizations, people, and companies that will pay handsomely for personalized data, particularly of the medical variety, to increase their own marketing efficiency and investments, thereby increasing their own profits. Protecting the people who entrust you with their data is essential to prevent hostile or malicious entities from causing harm or targeted advantageous usage of said data against they who gave it to you in the first place.